# Results & Figures 

**Purpose:** Read saved results from upstream notebooks and generate all figures and tables for the manuscript. No model fitting or heavy computation — only loading, formatting, and plotting.

**Data sources:**
- `data/processed/component_results.h5` — per-component projections, observations, posteriors
- `data/processed/slr_processed_data.h5` — historical observations, kinematics
- `data/processed/bayesian_ratestate_results.json` — semi-empirical model summary
- `data/processed/manuscript_headline_stats.json` — precomputed headline statistics
- `data/processed/ipcc_distributions.h5` — IPCC AR6 samples for comparison

**Sections mirror the manuscript:**
- §0 Setup & data loading
- §1 Headline numbers (Abstract & key claims)
- §2 Figure 1 — Observations
- §3 Figure 2 — Semi-empirical diagnostics
- §4 Figure 3 — Per-component projections vs IPCC
- §5 Figure 4 — Total GMSL forecast
- §6 Figure 5 — Uncertainty structure & exceedance
- §7 Budget closure diagnostics
- §8 Supplementary tables & figures

---
## §0 Setup & data loading

In [1]:
import json
import sys
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Project imports
sys.path.insert(0, str(Path('.').resolve()))
from component_io import (
    load_all_projections, load_component, list_components,
    PROJ_YEARS, PROJ_SSPS, DEFAULT_H5_PATH, N_SAMPLES, BASELINE_YEAR,
)
from slr_forecast.config import PROCESSED_DATA_DIR, FIG_DIR, SSPS
from component_projections import (
    read_ipcc_component_nc, ipcc_extract, read_ismip6_regional,
)

# Paths
H5_OBS = PROCESSED_DATA_DIR / 'slr_processed_data.h5'
H5_COMP = DEFAULT_H5_PATH
H5_IPCC = PROCESSED_DATA_DIR / 'ipcc_distributions.h5'
H5_RATESTATE_POST = PROCESSED_DATA_DIR / 'bayesian_ratestate_posterior.h5'
JSON_RATESTATE = PROCESSED_DATA_DIR / 'bayesian_ratestate_results.json'
JSON_HEADLINE = PROCESSED_DATA_DIR / 'manuscript_headline_stats.json'
JSON_GREENLAND_CT = PROCESSED_DATA_DIR / 'greenland_ct_estimate.json'

RAW_DIR = Path('..') / 'data' / 'raw'
CONF_BASE = str(RAW_DIR / 'ipcc_ar6' / 'slr' / 'ar6' / 'global' / 'confidence_output_files')
ISMIP6_BASE = str(RAW_DIR / 'ice_sheets' / 'ismip6' / 'ComputedScalarsPaper')
SSP_TO_CODE = {
    'SSP1-2.6': 'ssp126', 'SSP2-4.5': 'ssp245',
    'SSP3-7.0': 'ssp370', 'SSP5-8.5': 'ssp585',
}

# Plotting defaults
sys.path.insert(0, str(Path('.').resolve() / 'arete_mpl'))
import arete_mpl
arete_mpl.use('poster')
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
})

# Conversion
M_TO_MM = 1000.0
M_TO_CM = 100.0

# SSP colors (consistent across figures)
SSP_COLORS = {
    'SSP1-2.6': '#2166ac',
    'SSP2-4.5': '#1b7837',
    'SSP3-7.0': '#e08214',
    'SSP5-8.5': '#b2182b',
}

# Policy-relevant SSPs (exclude SSP5-8.5 for headline stats)
POLICY_SSPS = ['SSP1-2.6', 'SSP2-4.5', 'SSP3-7.0']

# Components included in the forecast sum (EAIS excluded)
FORECAST_COMPONENTS = ['ocean', 'glacier', 'greenland', 'apeninsula', 'wais']

print(f'Baseline year: {BASELINE_YEAR}')
print(f'Ensemble size: {N_SAMPLES}')
print(f'Projection years: {PROJ_YEARS[0]:.0f}–{PROJ_YEARS[-1]:.0f}')

Baseline year: 2005.0
Ensemble size: 2000
Projection years: 1950–2150


In [2]:
# Load all component projections
proj_years, all_proj = load_all_projections()
list_components()

# Index for year 2100
i2100 = np.searchsorted(proj_years, 2100)
i2050 = np.searchsorted(proj_years, 2050)

print(f'\nIndex for 2100: {i2100} (year = {proj_years[i2100]:.0f})')
print(f'Index for 2050: {i2050} (year = {proj_years[i2050]:.0f})')

File: /Users/minchew/Dropbox/Documents/Research/SLR/forecasting/global_simple_v1/slr_forecast/data/processed/component_results.h5
  Baseline year: 2005.0
  N samples: 2000
  Proj years: 1950–2150

  apeninsula
    model: linear_dols
    SSPs: SSP1-2.6, SSP2-4.5, SSP3-7.0, SSP5-8.5
    full samples: True
    saved: 2026-04-06T22:37:58.031844+00:00
  blended
    model: unknown
    SSPs: 
    full samples: False
    saved: 2026-05-01 22:02:53.669863+00:00
  eais
    model: trend_only
    SSPs: SSP1-2.6, SSP2-4.5, SSP3-7.0, SSP5-8.5
    full samples: True
    saved: 2026-04-06T22:39:06.942984+00:00
  glacier
    model: linear_dols
    SSPs: SSP1-2.6, SSP2-4.5, SSP3-7.0, SSP5-8.5
    full samples: True
    saved: 2026-04-19T03:10:05.150592+00:00
  greenland
    model: smb_literature_plus_discharge_delay
    SSPs: SSP1-2.6, SSP2-4.5, SSP3-7.0, SSP5-8.5
    full samples: True
    saved: 2026-05-01T07:00:06.265095+00:00
  ocean
    model: twolayer_noaa
    SSPs: SSP1-2.6, SSP2-4.5, SSP3-7.0, S

In [3]:
# Load observational data
with pd.HDFStore(H5_OBS, mode='r') as store:
    df_frederikse = store['raw/df_frederikse']
    df_nasa = store['raw/df_nasa_gmsl']
    df_berkeley = store['raw/df_berkeley']
    # Kinematics (rates)
    kin_keys = [k for k in store.keys() if 'kinematics' in k]
    print('Kinematics keys:', kin_keys)

# Helper: extract year range from DatetimeIndex or float index
def _year_range(idx):
    if hasattr(idx, 'year'):
        return f'{idx[0].year}–{idx[-1].year}'
    return f'{idx[0]:.1f}–{idx[-1]:.1f}'

print(f'Frederikse: {_year_range(df_frederikse.index)}')
print(f'NASA altimetry: {_year_range(df_nasa.index)}')
print(f'Berkeley Earth: {_year_range(df_berkeley.index)}')
print(f'Frederikse columns: {list(df_frederikse.columns[:6])}...')

Kinematics keys: ['/kinematics/gmst/berkeley', '/kinematics/gmsl/dangendorf', '/kinematics/gmsl/frederikse', '/kinematics/gmsl/frederikse_thermodynamic_gmsl', '/kinematics/gmsl/frederikse_thermodynamic_sum', '/kinematics/gmsl/horwath', '/kinematics/gmsl/ipcc', '/kinematics/gmsl/nasa']
Frederikse: 1900–2018
NASA altimetry: 1993–2025
Berkeley Earth: 1850–2024
Frederikse columns: ['year', 'gmsl_lower', 'gmsl', 'gmsl_upper', 'sum_contributors_lower', 'sum_contributors']...


In [4]:
# Load rate-and-state results
with open(JSON_RATESTATE) as f:
    ratestate = json.load(f)

# Load headline stats
with open(JSON_HEADLINE) as f:
    headline = json.load(f)

print('Rate-and-state R²:', ratestate['calibration']['r_squared'])
print('Rate-and-state projections at 2100 (mm):')
for ssp, vals in ratestate['projections_at_2100_mm'].items():
    print(f"  {ssp}: median={vals['median']:.0f}, 90% CI=[{vals['p5']:.0f}, {vals['p95']:.0f}]")

Rate-and-state R²: 0.968095506444475
Rate-and-state projections at 2100 (mm):
  SSP1-2.6: median=778, 90% CI=[643, 905]
  SSP2-4.5: median=1299, 90% CI=[1021, 1558]
  SSP3-7.0: median=1987, 90% CI=[1520, 2429]
  SSP5-8.5: median=2756, 90% CI=[2077, 3401]


In [5]:
# --- Load blended forecast samples (Gap 1) ---
# Exported by component_forecast.ipynb
blended = {}
with h5py.File(H5_COMP, 'r') as hf:
    if 'blended' in hf:
        bg = hf['blended']
        blended_years = bg['forecast_years'][:]
        blended_w = bg['sigmoid_weight'][:]
        blended_meta = dict(bg.attrs)
        for ssp in PROJ_SSPS:
            if ssp in bg:
                sg = bg[ssp]
                blended[ssp] = {
                    'samples': sg['samples'][:],
                    'median': sg['median'][:],
                    'p5': sg['p5'][:],
                    'p17': sg['p17'][:],
                    'p83': sg['p83'][:],
                    'p95': sg['p95'][:],
                }
        print(f'Blended forecast loaded: {sorted(blended.keys())}')
        print(f'  Years: {blended_years[0]:.1f}–{blended_years[-1]:.0f}')
        print(f'  Sigmoid center: {blended_meta.get("t_center", "?")}')
        print(f'  Components: {list(blended_meta.get("components_summed", []))}')
    else:
        print('WARNING: blended group not found in component_results.h5')
        print('  Run component_forecast.ipynb to export blended samples.')

# Index for year 2100 in blended forecast
i2100_b = np.argmin(np.abs(blended_years - 2100))
i2050_b = np.argmin(np.abs(blended_years - 2050))
print(f'  Blended index for 2100: {i2100_b} (year = {blended_years[i2100_b]:.0f})')

# --- Pre-industrial offset (from blended HDF5 attrs, established upstream) ---
# This is SLR from pre-industrial (~1850-1900) to the 2005 baseline.
# Value set in component_forecast.ipynb from Frederikse et al. (2020) + literature.
PREINDUSTRIAL_OFFSET_M = float(blended_meta['preindustrial_to_baseline_m'])
print(f'\nPre-industrial offset (from blended attrs): {PREINDUSTRIAL_OFFSET_M} m')

# --- Load IPCC total medians from NetCDF (Gap 2) ---
IPCC_MEDIANS_MM = {}
for ssp, code in SSP_TO_CODE.items():
    data = read_ipcc_component_nc(CONF_BASE, 'medium_confidence', code, 'total')
    if data is not None:
        ie = ipcc_extract(data)
        idx_2100 = np.argmin(np.abs(ie['years'] - 2100))
        IPCC_MEDIANS_MM[ssp] = ie['q50'][idx_2100]  # mm

print(f'\nIPCC AR6 total medians at 2100 (mm, from NetCDF):')
for ssp, val in IPCC_MEDIANS_MM.items():
    print(f'  {ssp}: {val:.0f} mm')

# --- Load rate-and-state posterior and sensitivity curves (Gaps 5, 7) ---
rs_posterior = {}
if H5_RATESTATE_POST.exists():
    with h5py.File(H5_RATESTATE_POST, 'r') as hf:
        # Posterior samples
        rs_posterior['coefficients'] = hf['posterior/coefficients'][:]
        rs_posterior['tau'] = hf['posterior/tau'][:]

        # Rate-vs-T curve
        rs_posterior['T_grid'] = hf['rate_vs_T/T_grid_C'][:]
        rs_posterior['rate_median'] = hf['rate_vs_T/rate_median_mm_yr'][:]
        rs_posterior['rate_p5'] = hf['rate_vs_T/rate_p5_mm_yr'][:]
        rs_posterior['rate_p95'] = hf['rate_vs_T/rate_p95_mm_yr'][:]

        # Sensitivity curve
        rs_posterior['sens_median'] = hf['sensitivity_vs_T/sensitivity_median_mm_yr_C'][:]
        rs_posterior['sens_p5'] = hf['sensitivity_vs_T/sensitivity_p5_mm_yr_C'][:]
        rs_posterior['sens_p95'] = hf['sensitivity_vs_T/sensitivity_p95_mm_yr_C'][:]

        # Record-averaged sensitivity
        rs_posterior['avg_sens_median'] = float(
            hf['record_averaged_sensitivity/median_mm_yr_C'][()])
        rs_posterior['avg_sens_p5'] = float(
            hf['record_averaged_sensitivity/p5_mm_yr_C'][()])
        rs_posterior['avg_sens_p95'] = float(
            hf['record_averaged_sensitivity/p95_mm_yr_C'][()])

    print(f'\nRate-and-state posterior loaded:')
    print(f'  Posterior samples: {rs_posterior["coefficients"].shape}')
    print(f'  Record-averaged sensitivity: {rs_posterior["avg_sens_median"]:.1f} '
          f'[{rs_posterior["avg_sens_p5"]:.1f}, {rs_posterior["avg_sens_p95"]:.1f}] '
          f'mm/yr/°C')
    print(f'  (cf. Rahmstorf 2007: 3.4 mm/yr/°C)')
else:
    print('\nWARNING: bayesian_ratestate_posterior.h5 not found.')
    print('  Run bayesian_ratestate.ipynb to export posterior samples.')

Blended forecast loaded: ['SSP1-2.6', 'SSP2-4.5', 'SSP3-7.0', 'SSP5-8.5']
  Years: 2026.0–2150
  Sigmoid center: 2035.0
  Components: ['ocean', 'glacier', 'greenland', 'apeninsula', 'wais', 'tws']
  Blended index for 2100: 74 (year = 2100)

Pre-industrial offset (from blended attrs): 0.19 m

IPCC AR6 total medians at 2100 (mm, from NetCDF):
  SSP1-2.6: 437 mm
  SSP2-4.5: 556 mm
  SSP3-7.0: 678 mm
  SSP5-8.5: 766 mm

Rate-and-state posterior loaded:
  Posterior samples: (320000, 4)
  Record-averaged sensitivity: 3.5 [3.0, 4.2] mm/yr/°C
  (cf. Rahmstorf 2007: 3.4 mm/yr/°C)


In [6]:
# --- Component sum at 2100 (21st-century contributions, relative to 2005 baseline) ---
# Diagnostic: raw component sums (no blending, no TWS) vs IPCC
print('=== Component-sum projections at 2100 (mm, rel. to 2005 baseline) ===')
print('NOTE: These are raw component sums (no blending, no TWS). See blended forecast below.')
print(f'{"SSP":<12} {"Median":>8} {"P5":>8} {"P95":>8} {"IPCC med":>10} {"Ratio":>8}')
print('-' * 60)

for ssp in PROJ_SSPS:
    total_samples = np.zeros(N_SAMPLES)
    for comp in FORECAST_COMPONENTS:
        total_samples += all_proj[comp][ssp]['samples'][:, i2100]
    
    med = np.median(total_samples) * M_TO_MM
    p5 = np.percentile(total_samples, 5) * M_TO_MM
    p95 = np.percentile(total_samples, 95) * M_TO_MM
    ipcc = IPCC_MEDIANS_MM.get(ssp, np.nan)
    ratio = med / ipcc if not np.isnan(ipcc) else np.nan
    
    print(f'{ssp:<12} {med:8.0f} {p5:8.0f} {p95:8.0f} {ipcc:10.0f} {ratio:8.2f}x')

=== Component-sum projections at 2100 (mm, rel. to 2005 baseline) ===
NOTE: These are raw component sums (no blending, no TWS). See blended forecast below.
SSP            Median       P5      P95   IPCC med    Ratio
------------------------------------------------------------
SSP1-2.6          740      418     1855        437     1.69x
SSP2-4.5          913      584     2019        556     1.64x
SSP3-7.0         1124      771     2227        678     1.66x
SSP5-8.5         1321      953     2445        766     1.72x


In [7]:
# --- Blended forecast at 2100 (21st-century contributions, rel. to 2005 baseline) ---
# These are the authoritative manuscript values.
print('=== Blended forecast at 2100 (cm, rel. to 2005 baseline) ===')
print()

for ssp in PROJ_SSPS:
    s = blended[ssp]['samples'][:, i2100_b]
    med = np.median(s) * M_TO_CM
    p5 = np.percentile(s, 5) * M_TO_CM
    p95 = np.percentile(s, 95) * M_TO_CM
    print(f'{ssp}: {med:.0f} cm [{p5:.0f}–{p95:.0f} cm]')

# Cross-check against headline stats
print()
print('Headline stats (manuscript_headline_stats.json):')
for ssp in PROJ_SSPS:
    hs = headline['scenarios'][ssp]['2100']
    print(f'  {ssp}: {hs["median_mm_rel_baseline"]/10:.0f} cm '
          f'[{hs["p5_mm_rel_baseline"]/10:.0f}–{hs["p95_mm_rel_baseline"]/10:.0f} cm]')

=== Blended forecast at 2100 (cm, rel. to 2005 baseline) ===

SSP1-2.6: 75 cm [43–183 cm]
SSP2-4.5: 91 cm [59–200 cm]
SSP3-7.0: 111 cm [78–221 cm]
SSP5-8.5: 131 cm [95–238 cm]

Headline stats (manuscript_headline_stats.json):
  SSP1-2.6: 75 cm [42–183 cm]
  SSP2-4.5: 91 cm [59–200 cm]
  SSP3-7.0: 111 cm [78–221 cm]
  SSP5-8.5: 131 cm [94–238 cm]


In [8]:
# --- Pre-industrial values (for abstract) ---
# Uses blended samples + authoritative pre-industrial offset
print(f'=== Abstract values (m, rel. to pre-industrial, offset = {PREINDUSTRIAL_OFFSET_M} m) ===')
print()

for ssp in POLICY_SSPS:
    s = blended[ssp]['samples'][:, i2100_b]
    s_pi = s + PREINDUSTRIAL_OFFSET_M
    
    med = np.median(s_pi)
    p5 = np.percentile(s_pi, 5)
    p95 = np.percentile(s_pi, 95)
    
    print(f'{ssp}: median = {med:.2f} m [{p5:.2f}–{p95:.2f} m]')

# Cross-check against headline stats
print()
print('Headline stats:')
for ssp in POLICY_SSPS:
    hs = headline['scenarios'][ssp]['2100']
    print(f'  {ssp}: {hs["median_m_rel_preindustrial"]:.2f} m '
          f'[{hs["p5_m_rel_preindustrial"]:.2f}–{hs["p95_m_rel_preindustrial"]:.2f} m]')
print(f'  Range: {headline["preindustrial_median_range_m"]} m')

=== Abstract values (m, rel. to pre-industrial, offset = 0.19 m) ===

SSP1-2.6: median = 0.94 m [0.62–2.02 m]
SSP2-4.5: median = 1.10 m [0.78–2.19 m]
SSP3-7.0: median = 1.30 m [0.97–2.40 m]

Headline stats:
  SSP1-2.6: 0.94 m [0.62–2.02 m]
  SSP2-4.5: 1.10 m [0.78–2.19 m]
  SSP3-7.0: 1.30 m [0.97–2.40 m]
  Range: [0.94, 1.3] m


In [9]:
# --- Exceedance probabilities ---
# Uses blended samples + authoritative pre-industrial offset
print('=== P(exceed 1 m relative to pre-industrial) ===')
print()

exceed_probs = {}
for ssp in POLICY_SSPS:
    s = blended[ssp]['samples'][:, i2100_b]
    s_pi = s + PREINDUSTRIAL_OFFSET_M
    p_exceed = np.mean(s_pi > 1.0) * 100
    exceed_probs[ssp] = p_exceed
    print(f'{ssp}: P(>1 m) = {p_exceed:.1f}%')

avg_exceed = np.mean(list(exceed_probs.values()))
print(f'\nScenario-averaged: {avg_exceed:.1f}%')

# Cross-check against headline stats
print()
print('Headline stats:')
for ssp in POLICY_SSPS:
    hs = headline['scenarios'][ssp]['2100']
    print(f'  {ssp}: P(>1 m) = {hs["P_exceed_1.0m_preindustrial"]:.1f}%')
print(f'  Avg: {headline["P_exceed_1m_avg_policy_ssps"]:.1f}%')

=== P(exceed 1 m relative to pre-industrial) ===

SSP1-2.6: P(>1 m) = 42.7%
SSP2-4.5: P(>1 m) = 68.2%
SSP3-7.0: P(>1 m) = 92.5%

Scenario-averaged: 67.8%

Headline stats:
  SSP1-2.6: P(>1 m) = 42.7%
  SSP2-4.5: P(>1 m) = 68.2%
  SSP3-7.0: P(>1 m) = 92.5%
  Avg: 67.8%


In [10]:
# --- IPCC comparison ratios (blended forecast) ---
print('=== Our median / IPCC median (blended forecast) ===')
print()

for ssp in PROJ_SSPS:
    s = blended[ssp]['samples'][:, i2100_b]
    med_mm = np.median(s) * M_TO_MM
    ipcc = IPCC_MEDIANS_MM[ssp]
    ratio = med_mm / ipcc
    pct_above = (ratio - 1) * 100
    
    print(f'{ssp}: {med_mm:.0f} / {ipcc:.0f} = {ratio:.2f}x ({pct_above:.0f}% above IPCC)')

# Cross-check against headline stats
print()
print('Headline stats:')
for ssp in PROJ_SSPS:
    if ssp in headline.get('ipcc_comparison', {}):
        ic = headline['ipcc_comparison'][ssp]
        print(f'  {ssp}: {ic["pct_above_ipcc"]:.0f}% above')
print(f'  Policy SSP range: {headline.get("ipcc_pct_above_range", "N/A")}%')

=== Our median / IPCC median (blended forecast) ===

SSP1-2.6: 751 / 437 = 1.72x (72% above IPCC)
SSP2-4.5: 913 / 556 = 1.64x (64% above IPCC)
SSP3-7.0: 1112 / 678 = 1.64x (64% above IPCC)
SSP5-8.5: 1306 / 766 = 1.70x (70% above IPCC)

Headline stats:
  SSP1-2.6: 71% above
  SSP2-4.5: 63% above
  SSP3-7.0: 64% above
  SSP5-8.5: 57% above
  Policy SSP range: [63, 71]%


In [11]:
# --- Variance decomposition (blended forecast) ---
print('=== Variance decomposition (law of total variance across policy SSPs) ===')
print()

# Collect 2100 blended samples per SSP
ssp_samples = {}
for ssp in POLICY_SSPS:
    ssp_samples[ssp] = blended[ssp]['samples'][:, i2100_b]

# Within-scenario variance (average of per-SSP variances)
var_within = np.mean([np.var(s) for s in ssp_samples.values()])

# Between-scenario variance (variance of per-SSP means)
var_between = np.var([np.mean(s) for s in ssp_samples.values()])

var_total = var_within + var_between
pct_within = var_within / var_total * 100
pct_between = var_between / var_total * 100

print(f'Within-scenario:  {pct_within:.1f}% of total variance')
print(f'Between-scenario: {pct_between:.1f}% of total variance')

# SSP2-4.5 90% range
ssp245 = ssp_samples['SSP2-4.5']
range_90 = (np.percentile(ssp245, 95) - np.percentile(ssp245, 5)) * M_TO_CM
median_spread = (np.median(ssp_samples['SSP3-7.0']) - np.median(ssp_samples['SSP1-2.6'])) * M_TO_CM
print(f'\nSSP2-4.5 90% range: {range_90:.0f} cm')
print(f'Cross-scenario median spread: {median_spread:.0f} cm')
print(f'Ratio: {range_90 / median_spread:.0f}x')

# Cross-check against headline stats
vd = headline['variance_decomposition']
print(f'\nHeadline stats:')
print(f'  Within: {vd["pct_within_scenario"]:.1f}%, Between: {vd["pct_between_scenario"]:.1f}%')
print(f'  90% range SSP2-4.5: {headline["within_scenario_90pct_range_mm_SSP245"]/10:.0f} cm')
print(f'  Cross-scenario spread: {headline["cross_scenario_median_spread_mm"]/10:.0f} cm')
print(f'  Ratio: {headline["ratio_within_to_across"]:.0f}x')

=== Variance decomposition (law of total variance across policy SSPs) ===

Within-scenario:  92.5% of total variance
Between-scenario: 7.5% of total variance

SSP2-4.5 90% range: 141 cm
Cross-scenario median spread: 36 cm
Ratio: 4x

Headline stats:
  Within: 92.5%, Between: 7.5%
  90% range SSP2-4.5: 141 cm
  Cross-scenario spread: 36 cm
  Ratio: 4x


---
## §2 Figure 1 — Observations

Three panels:
- **(A)** GMSL (Frederikse + NASA) and GMST (Berkeley Earth) since 1900
- **(B)** GMSL rate vs GMST with rate-and-state model fit
- **(C)** WAIS: IMBIE observations vs ISMIP6 models

In [12]:
# --- Panel B: Rate vs temperature scatter + rate-and-state fit ---
# Rate-and-state coefficients (from JSON)
rs_coeff = ratestate['calibration']['coefficients_mm_yr']
a_rs = rs_coeff['dalpha_dT']   # mm/yr/°C²
b_rs = rs_coeff['alpha0']      # mm/yr/°C
c_rs = rs_coeff['trend']       # mm/yr

print('Rate-and-state coefficients:')
print(f'  a (dα/dT)  = {a_rs:.3f} mm/yr/°C²')
print(f'  b (α₀)     = {b_rs:.3f} mm/yr/°C')
print(f'  c (trend)  = {c_rs:.3f} mm/yr')

# Instantaneous sensitivity: dRate/dT = 2aT + b
T_epochs = {'1950 (T≈-0.4°C)': -0.4, '2000 (T≈0°C)': 0.0, '2019 (T≈+0.24°C)': 0.244}
print('\nInstantaneous sensitivity dRate/dT = 2aT + b:')
for label, T in T_epochs.items():
    sens = 2 * a_rs * T + b_rs
    print(f'  {label}: {sens:.1f} mm/yr/°C')

# Record-averaged sensitivity (from posterior if available, else from coefficients)
if rs_posterior:
    print(f'\nRecord-averaged sensitivity (from posterior, with uncertainty):')
    print(f'  {rs_posterior["avg_sens_median"]:.1f} '
          f'[{rs_posterior["avg_sens_p5"]:.1f}, {rs_posterior["avg_sens_p95"]:.1f}] '
          f'mm/yr/°C')
else:
    T_start, T_end = -0.6, 0.244
    avg_sens = a_rs * (T_start + T_end) + b_rs
    print(f'\nRecord-averaged sensitivity (from median coefficients):')
    print(f'  {avg_sens:.1f} mm/yr/°C')
print(f'Rahmstorf (2007): 3.4 mm/yr/°C')

Rate-and-state coefficients:
  a (dα/dT)  = 4.552 mm/yr/°C²
  b (α₀)     = 5.170 mm/yr/°C
  c (trend)  = 2.661 mm/yr

Instantaneous sensitivity dRate/dT = 2aT + b:
  1950 (T≈-0.4°C): 1.5 mm/yr/°C
  2000 (T≈0°C): 5.2 mm/yr/°C
  2019 (T≈+0.24°C): 7.4 mm/yr/°C

Record-averaged sensitivity (from posterior, with uncertainty):
  3.5 [3.0, 4.2] mm/yr/°C
Rahmstorf (2007): 3.4 mm/yr/°C


In [13]:
# --- Panel C: WAIS observations vs ISMIP6 ---
# Load IMBIE WAIS observations
wais = load_component('wais')
wais_obs = wais['observations']
print(f"WAIS obs years: {wais_obs['years'][0]:.0f}–{wais_obs['years'][-1]:.0f}")
print(f"WAIS obs final value: {wais_obs['H_obs'][-1] * M_TO_MM:.1f} mm")

# Load ISMIP6 WAIS trajectories (region=1) from raw data
try:
    ismip6_wais = read_ismip6_regional(ISMIP6_BASE, region=1)
    n_runs = len(ismip6_wais)
    print(f'\nISMIP6 WAIS runs loaded: {n_runs}')

    # Collect endpoint values (at ~2020 or latest available)
    # read_ismip6_regional returns dicts with keys 'time' and 'sle_m'
    ismip6_endpoints = []
    for key, traj in ismip6_wais.items():
        years = traj['time']
        vals_mm = traj['sle_m'] * M_TO_MM  # convert m to mm SLE
        idx_2020 = np.argmin(np.abs(years - 2020))
        ismip6_endpoints.append(vals_mm[idx_2020])

    ismip6_endpoints = np.array(ismip6_endpoints)
    n_reproduce = np.sum(ismip6_endpoints >= wais_obs['H_obs'][-1] * M_TO_MM)
    n_gain_mass = np.sum(ismip6_endpoints < 0)
    print(f'  At 2020: median = {np.median(ismip6_endpoints):.1f} mm, '
          f'observed = {wais_obs["H_obs"][-1] * M_TO_MM:.1f} mm')
    print(f'  Runs reproducing observed: {n_reproduce}/{n_runs}')
    print(f'  Runs showing mass gain: {n_gain_mass}/{n_runs} '
          f'({n_gain_mass/n_runs*100:.0f}%)')
except Exception as e:
    print(f'\nCould not load ISMIP6 data: {e}')
    import traceback; traceback.print_exc()
    print('  Check ISMIP6_BASE path and read_ismip6_regional return format.')

WAIS obs years: 1992–2020
WAIS obs final value: 5.0 mm

ISMIP6 WAIS runs loaded: 77
  At 2020: median = 0.7 mm, observed = 5.0 mm
  Runs reproducing observed: 6/77
  Runs showing mass gain: 18/77 (23%)


In [14]:
# --- Semi-empirical model comparison table ---
print('=== Semi-empirical projections at 2100 (21st-century rise, SSP2-4.5) ===')
print()
print(f'{"Model":<30} {"Median (m)":>12} {"90% CI":>18}')
print('-' * 62)
print(f'{"Rahmstorf (2007)":<30} {0.67:>12.2f} {"0.63–0.71":>18}')
print(f'{"Vermeer & Rahmstorf (2009)":<30} {0.96:>12.2f} {"0.80–1.12":>18}')

rs_ssp245 = ratestate['projections_at_2100_mm']['SSP2-4.5']
rs_med = rs_ssp245['median'] / M_TO_MM
rs_p5 = rs_ssp245['p5'] / M_TO_MM
rs_p95 = rs_ssp245['p95'] / M_TO_MM
print(f'{"This study (rate-and-state)":<30} {rs_med:>12.2f} {f"{rs_p5:.2f}–{rs_p95:.2f}":>18}')
print()
print(f'Ratio to R07: {rs_med / 0.67:.1f}x')
print(f'Ratio to VR09: {rs_med / 0.96:.1f}x ({(rs_med / 0.96 - 1) * 100:.0f}% above)')

=== Semi-empirical projections at 2100 (21st-century rise, SSP2-4.5) ===

Model                            Median (m)             90% CI
--------------------------------------------------------------
Rahmstorf (2007)                       0.67          0.63–0.71
Vermeer & Rahmstorf (2009)             0.96          0.80–1.12
This study (rate-and-state)            1.30          1.02–1.56

Ratio to R07: 1.9x
Ratio to VR09: 1.4x (35% above)


In [15]:
# --- Rate-and-state projections across all SSPs ---
print('=== Rate-and-state projections at 2100 (mm) ===')
print()
for ssp in PROJ_SSPS:
    vals = ratestate['projections_at_2100_mm'][ssp]
    print(f"{ssp}: median={vals['median']:.0f} mm [{vals['p5']:.0f}–{vals['p95']:.0f}]")

=== Rate-and-state projections at 2100 (mm) ===

SSP1-2.6: median=778 mm [643–905]
SSP2-4.5: median=1299 mm [1021–1558]
SSP3-7.0: median=1987 mm [1520–2429]
SSP5-8.5: median=2756 mm [2077–3401]


---
## §4 Figure 3 — Per-component projections vs IPCC

Individual sea-level contributors showing our projections overlaid with IPCC AR6 medians and observations.

In [16]:
# --- Per-component summary table at 2100 ---
ALL_COMPONENTS = ['ocean', 'glacier', 'greenland', 'apeninsula', 'wais', 'eais']

print('=== Per-component projections at 2100 (mm, SSP2-4.5) ===')
print(f'{"Component":<15} {"Median":>8} {"P5":>8} {"P95":>8} {"In forecast?":>14}')
print('-' * 55)

for comp in ALL_COMPONENTS:
    data = all_proj[comp]['SSP2-4.5']
    samples = data['samples'][:, i2100]
    med = np.median(samples) * M_TO_MM
    p5 = np.percentile(samples, 5) * M_TO_MM
    p95 = np.percentile(samples, 95) * M_TO_MM
    in_forecast = 'Yes' if comp in FORECAST_COMPONENTS else 'No'
    print(f'{comp:<15} {med:8.0f} {p5:8.0f} {p95:8.0f} {in_forecast:>14}')

=== Per-component projections at 2100 (mm, SSP2-4.5) ===
Component         Median       P5      P95   In forecast?
-------------------------------------------------------
ocean                236      198      284            Yes
glacier              127      119      136            Yes
greenland            144       97      189            Yes
apeninsula             9        4       15            Yes
wais                 390       67     1507            Yes
eais                 -19      -29       -9             No


In [17]:
# --- Key component parameters ---
print('=== Key fitted parameters ===')
print()

# Glacier sensitivity
glacier = load_component('glacier')
print(f"Glacier R²: {glacier['metadata'].get('r2', 'N/A')}")
print(f"Glacier model type: {glacier['metadata'].get('model_type', 'N/A')}")
print()

# Greenland
greenland = load_component('greenland')
if 'posteriors' in greenland and 'discharge' in greenland['posteriors']:
    discharge = greenland['posteriors']['discharge']
    print(f"Greenland discharge R²: {greenland['metadata'].get('r2_dyn', 'N/A')}")
print()

# WAIS
print(f"WAIS model type: {wais['metadata'].get('model_type', 'N/A')}")
print(f"WAIS rheology factor: {wais['metadata'].get('rheology_factor_median', 'N/A')}")
print(f"WAIS SSP-independent: {wais['metadata'].get('ssp_independent', 'N/A')}")

=== Key fitted parameters ===

Glacier R²: 0.996379517197939
Glacier model type: linear_dols

Greenland discharge R²: N/A

WAIS model type: a4_deep_uncertainty
WAIS rheology factor: 1.28
WAIS SSP-independent: True


In [18]:
# TODO: Multi-panel figure showing each component's projection time series
#       with observations and IPCC comparison
#       Layout: 2x3 or 3x2 grid for ocean, glacier, greenland, wais, eais, peninsula

print('TODO: Per-component projection figure')

TODO: Per-component projection figure


In [19]:
# --- Blended vs raw component sum comparison ---
# Diagnostic: quantify the effect of rate-space blending
print('=== Blended vs raw component sum at 2100 (cm, rel. to 2005) ===')
print(f'{"SSP":<12} {"Blended":>10} {"Raw sum":>10} {"Diff":>8}')
print('-' * 42)

for ssp in PROJ_SSPS:
    # Blended
    s_bl = blended[ssp]['samples'][:, i2100_b]
    med_bl = np.median(s_bl) * M_TO_CM
    
    # Raw component sum
    total_raw = np.zeros(N_SAMPLES)
    for comp in FORECAST_COMPONENTS:
        total_raw += all_proj[comp][ssp]['samples'][:, i2100]
    med_raw = np.median(total_raw) * M_TO_CM
    
    print(f'{ssp:<12} {med_bl:10.0f} {med_raw:10.0f} {med_bl - med_raw:8.1f}')

print()
print('Blended includes TWS and rate-space blending with satellite quadratic.')

=== Blended vs raw component sum at 2100 (cm, rel. to 2005) ===
SSP             Blended    Raw sum     Diff
------------------------------------------
SSP1-2.6             75         74      1.1
SSP2-4.5             91         91     -0.1
SSP3-7.0            111        112     -1.1
SSP5-8.5            131        132     -1.5

Blended includes TWS and rate-space blending with satellite quadratic.


In [20]:
# TODO: Total GMSL forecast figure
#       - NASA altimetry observations (black)
#       - Blended forecasts per SSP (colored, 66% and 90% CI shading)
#       - Satellite quadratic extrapolation (grey dashed)
#       - IPCC AR6 medians (dashed colored)

print('TODO: Total GMSL forecast figure')

TODO: Total GMSL forecast figure


In [21]:
# --- Exceedance thresholds (blended forecast) ---
print('=== Exceedance thresholds at 2100 (rel. to pre-industrial, blended) ===')
print()

thresholds_m = [0.5, 1.0, 1.5, 2.0]
print(f'{"SSP":<12}', '  '.join([f'P(>{t:.1f}m)' for t in thresholds_m]))
print('-' * 60)

for ssp in POLICY_SSPS:
    s = blended[ssp]['samples'][:, i2100_b]
    s_pi = s + PREINDUSTRIAL_OFFSET_M
    
    probs = [np.mean(s_pi > t) * 100 for t in thresholds_m]
    print(f'{ssp:<12}', '  '.join([f'{p:7.1f}%' for p in probs]))

# Cross-check against headline stats
print()
print('Headline stats:')
for ssp in POLICY_SSPS:
    hs = headline['scenarios'][ssp]['2100']
    probs = [hs[f'P_exceed_{t:.1f}m_preindustrial'] for t in thresholds_m]
    print(f'  {ssp:<12}', '  '.join([f'{p:7.1f}%' for p in probs]))

=== Exceedance thresholds at 2100 (rel. to pre-industrial, blended) ===

SSP          P(>0.5m)  P(>1.0m)  P(>1.5m)  P(>2.0m)
------------------------------------------------------------
SSP1-2.6       100.0%     42.7%     16.2%      5.4%
SSP2-4.5       100.0%     68.2%     21.6%      8.6%
SSP3-7.0       100.0%     92.5%     31.1%     12.3%

Headline stats:
  SSP1-2.6       100.0%     42.7%     16.2%      5.4%
  SSP2-4.5       100.0%     68.2%     21.6%      8.6%
  SSP3-7.0       100.0%     92.5%     31.1%     12.3%


---
## §7 Budget closure diagnostics

Component sum vs NASA altimetry over the satellite era (1993–2020). Residual attribution across components.

In [22]:
# --- Budget closure ---
# TODO: Compute component sum over satellite era and compare to NASA altimetry.
#       This requires hindcast values (observations) from each component,
#       not the MC projection samples.
#       Values from manuscript:
#         Residual rate: 0.37 mm/yr
#         Thermosteric absorbs: 57%
#         TWS absorbs: 23%
#         All ice components within 1.5 sigma

print('TODO: Budget closure diagnostic')
print()
print('Manuscript values:')
print('  Residual rate trend: 0.37 mm/yr')
print('  Thermosteric share: 57%')
print('  TWS share: 23%')
print('  All ice components within 1.5σ')

TODO: Budget closure diagnostic

Manuscript values:
  Residual rate trend: 0.37 mm/yr
  Thermosteric share: 57%
  TWS share: 23%
  All ice components within 1.5σ


---
## §8 Supplementary tables & figures

In [23]:
# --- Table S1: Observational datasets ---
# TODO: Generate table of calibration datasets, time spans, and sources
#       Read from component metadata

print('=== Table S1: Calibration datasets ===')
print(f'{"Component":<15} {"Type":<30} {"Obs years":<20}')
print('-' * 65)

for comp_name in ALL_COMPONENTS:
    comp = load_component(comp_name)
    mtype = comp['metadata'].get('model_type', 'N/A')
    obs = comp.get('observations', {})
    
    # Handle different obs structures
    if 'years' in obs:
        yr_range = f"{obs['years'][0]:.0f}–{obs['years'][-1]:.0f}"
    elif 'smb' in obs:  # Greenland has sub-components
        smb_yrs = obs['smb']['years']
        dyn_yrs = obs['discharge']['years']
        yr_range = f"SMB: {smb_yrs[0]:.0f}–{smb_yrs[-1]:.0f}, D: {dyn_yrs[0]:.0f}–{dyn_yrs[-1]:.0f}"
    else:
        yr_range = 'N/A'
    
    print(f'{comp_name:<15} {mtype:<30} {yr_range:<20}')

=== Table S1: Calibration datasets ===
Component       Type                           Obs years           
-----------------------------------------------------------------
ocean           twolayer_noaa                  1956–2026           
glacier         linear_dols                    2000–2024           
greenland       smb_literature_plus_discharge_delay SMB: 1972–2018, D: 1972–2018
apeninsula      linear_dols                    1992–2020           
wais            a4_deep_uncertainty            1992–2020           
eais            trend_only                     1992–2020           


---
## Gaps & TODOs

**Resolved:**
1. ~~Blended forecast samples~~ — Export cell added to `component_forecast.ipynb`; loaded from `component_results.h5` `blended/` group.
2. ~~IPCC medians~~ — Now read from IPCC AR6 NetCDF files via `read_ipcc_component_nc()`.
3. ~~Pre-industrial offset~~ — Computed from Frederikse reconstruction (GMSL at 2000 minus 1850–1900 mean).
4. ~~ISMIP6 trajectories~~ — Loaded from raw data via `read_ismip6_regional(region=1)`.
5. ~~Rate-and-state posterior~~ — Export cell added to `bayesian_ratestate.ipynb`; posterior samples, rate curve, and sensitivity curve saved to `bayesian_ratestate_posterior.h5`.
7. ~~Sensitivity evolution~~ — Saved as part of gap 5: equilibrium sensitivity curve with full posterior uncertainty, plus record-averaged sensitivity for Rahmstorf comparison.

**Remaining:**
6. **Budget closure computation** — TODO: The budget closure diagnostics (residual rate, per-component attribution) need careful review before implementing here. The calculation is in `component_summation.ipynb` cells 9 and 21, but the methodology is under active discussion. Defer until the approach is finalized.

**Before submission:**
- Run `component_forecast.ipynb` to populate `blended/` group in `component_results.h5`
- Run `bayesian_ratestate.ipynb` to populate `bayesian_ratestate_posterior.h5`
- Then run this notebook end-to-end to verify all numbers match the manuscript

---
## Remaining TODOs

1. **Budget closure computation** -- The budget closure diagnostics (residual rate, per-component attribution) need careful review before implementing here. The calculation is in `component_summation.ipynb` cells 9 and 21, but the methodology is under active discussion. Defer until the approach is finalized.
2. **Figures** -- Per-component projection figure (cell 21), total GMSL forecast figure (cell 24), exceedance curve plot (cell 25) are stubs.
3. **Manuscript text vs headline stats** -- The manuscript text (lines 105-107) reports values 1-3 cm / 1-3% below the current `manuscript_headline_stats.json`. Either the manuscript needs updating or the headline stats were regenerated after the last manuscript edit. Reconcile before submission.

**Before submission:**
- Run `component_forecast.ipynb` to populate `blended/` group in `component_results.h5`
- Run `bayesian_ratestate.ipynb` to populate `bayesian_ratestate_posterior.h5`
- Then run this notebook end-to-end to verify all numbers match the manuscript